In [1]:
import pandas as pd

df = pd.read_csv('global_power_plant_database.csv')
df.head()

/var/folders/6c/vj06ny152zq1_xz1kf1j7hk00000gn/T/ipykernel_41614/2544498401.py:3: DtypeWarning: Columns (0: other_fuel3) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('global_power_plant_database.csv')


,country,country_long,name,gppd_idnr,capacity_mw,latitude,longitude,primary_fuel,other_fuel1,other_fuel2,...,estimated_generation_gwh_2013,estimated_generation_gwh_2014,estimated_generation_gwh_2015,estimated_generation_gwh_2016,estimated_generation_gwh_2017,estimated_generation_note_2013,estimated_generation_note_2014,estimated_generation_note_2015,estimated_generation_note_2016,estimated_generation_note_2017
0,AFG,Afghanistan,Kajaki Hydroelectric Power Plant Afghanistan,GEODB0040538,33.0,32.322,65.1190,Hydro,NaN,NaN,...,123.77,162.90,97.39,137.76,119.50,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1
1,AFG,Afghanistan,Kandahar DOG,WKS0070144,10.0,31.670,65.7950,Solar,NaN,NaN,...,18.43,17.48,18.25,17.70,18.29,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE
2,AFG,Afghanistan,Kandahar JOL,WKS0071196,10.0,31.623,65.7920,Solar,NaN,NaN,...,18.64,17.58,19.10,17.62,18.72,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE,SOLAR-V1-NO-AGE
3,AFG,Afghanistan,Mahipar Hydroelectric Power Plant Afghanistan,GEODB0040541,66.0,34.556,69.4787,Hydro,NaN,NaN,...,225.06,203.55,146.90,230.18,174.91,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1
4,AFG,Afghanistan,Naghlu Dam Hydroelectric Power Plant Afghanistan,GEODB0040534,100.0,34.641,69.7170,Hydro,NaN,NaN,...,406.16,357.22,270.99,395.38,350.80,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1,HYDRO-V1


In [2]:
df = pd.read_csv('global_power_plant_database.csv', low_memory=False)
df.shape

(34936, 36)

In [3]:
df.isnull().sum()

country                               0
country_long                          0
name                                  0
gppd_idnr                             0
capacity_mw                           0
latitude                              0
longitude                             0
primary_fuel                          0
other_fuel1                       32992
other_fuel2                       34660
other_fuel3                       34844
commissioning_year                17489
owner                             14068
source                               15
url                                  18
geolocation_source                  419
wepp_id                           18702
year_of_capacity_data             20049
generation_gwh_2013               28519
generation_gwh_2014               27710
generation_gwh_2015               26733
generation_gwh_2016               25792
generation_gwh_2017               25436
generation_gwh_2018               25299
generation_gwh_2019               25277


In [4]:
df_clean = df[['country', 'country_long', 'name', 'capacity_mw', 
               'latitude', 'longitude', 'primary_fuel', 'commissioning_year']].copy()
df_clean.head()

,country,country_long,name,capacity_mw,latitude,longitude,primary_fuel,commissioning_year
0,AFG,Afghanistan,Kajaki Hydroelectric Power Plant Afghanistan,33.0,32.322,65.1190,Hydro,NaN
1,AFG,Afghanistan,Kandahar DOG,10.0,31.670,65.7950,Solar,NaN
2,AFG,Afghanistan,Kandahar JOL,10.0,31.623,65.7920,Solar,NaN
3,AFG,Afghanistan,Mahipar Hydroelectric Power Plant Afghanistan,66.0,34.556,69.4787,Hydro,NaN
4,AFG,Afghanistan,Naghlu Dam Hydroelectric Power Plant Afghanistan,100.0,34.641,69.7170,Hydro,NaN


In [10]:
fuel_map = {
    'Hydro': 'Renewable',
    'Solar': 'Renewable',
    'Wind': 'Renewable',
    'Nuclear': 'Renewable',
    'Biomass': 'Renewable',
    'Geothermal': 'Renewable',
    'Wave and Tidal': 'Renewable',
    'Coal': 'Fossil',
    'Gas': 'Fossil',
    'Oil': 'Fossil',
    'Petcoke': 'Fossil',
    'Cogeneration': 'Fossil',
    'Other': 'Other' ,
    'Waste': 'Other',
    'Storage': 'Other'
}

df_clean['fuel_category'] = df_clean['primary_fuel'].map(fuel_map)
df_clean['fuel_category'].value_counts()

fuel_category
Renewable    24989
Fossil        8701
Other         1246
Name: count, dtype: int64

In [7]:
df_clean['fuel_category'].head()   #my personal step...not recommended by claude...the code is correct

0    Renewable
1    Renewable
2    Renewable
3    Renewable
4    Renewable
Name: fuel_category, dtype: str

In [8]:
# Step 1: how many rows failed to categorize?
df_clean['fuel_category'].isnull().sum()

# Step 2: what fuel types are those failing rows?
df_clean[df_clean['fuel_category'].isnull()]['primary_fuel'].unique()

<ArrowStringArray>
['Waste', 'Storage']
Length: 2, dtype: str

In [11]:
df_clean['fuel_category'].isnull().sum()  #rechecking the count to see all the rows have been considered

np.int64(0)

In [16]:
df_clean.to_csv('power_plants_cleaned.csv', index=False)

In [13]:
!pip install sqlalchemy psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 11.1 MB/s  0:00:00 10.9 MB/s eta 0:00:01


In [14]:
from sqlalchemy import create_engine

engine = create_engine('postgresql://truptikotian@localhost:5432/truptikotian')

df_clean.to_sql('power_plants', engine, if_exists='replace', index=False)

936

In [15]:
import pandas as pd

pd.read_sql("SELECT country, capacity_mw, fuel_category FROM power_plants LIMIT 5;", engine)

,country,capacity_mw,fuel_category
0,AFG,33.0,Renewable
1,AFG,10.0,Renewable
2,AFG,10.0,Renewable
3,AFG,66.0,Renewable
4,AFG,100.0,Renewable


In [17]:
pd.read_sql("SELECT COUNT(*) FROM power_plants;", engine)

,count
0,34936


In [18]:
query1 = """
SELECT country_long, 
       COUNT(*) AS plant_count,
       SUM(capacity_mw) AS total_capacity_mw
FROM power_plants
WHERE fuel_category = 'Renewable' 
  AND commissioning_year >= 2010
GROUP BY country_long
ORDER BY total_capacity_mw DESC
LIMIT 10;
"""

pd.read_sql(query1, engine)

,country_long,plant_count,total_capacity_mw
0,United States of America,4038,111497.65000
1,China,37,68151.00000
2,Brazil,724,34906.02416
3,India,37,9844.52000
4,Vietnam,106,9738.20000
5,Iran,7,4531.00000
6,Germany,18,3621.06200
7,South Africa,53,3326.24000
8,Russia,16,3187.80000
9,Venezuela,2,3044.00000


In [20]:
query2 = """
SELECT country_long,
       ROUND((SUM(CASE WHEN fuel_category = 'Renewable' THEN capacity_mw ELSE 0 END) / SUM(capacity_mw) * 100)::numeric, 1) AS renewable_pct,
       ROUND((SUM(CASE WHEN fuel_category = 'Fossil' THEN capacity_mw ELSE 0 END) / SUM(capacity_mw) * 100)::numeric, 1) AS fossil_pct,
       SUM(capacity_mw) AS total_capacity_mw
FROM power_plants
GROUP BY country_long
HAVING SUM(capacity_mw) > 5000
ORDER BY renewable_pct DESC
LIMIT 15;
"""

pd.read_sql(query2, engine)

,country_long,renewable_pct,fossil_pct,total_capacity_mw
0,Paraguay,100.0,0.0,8760.00000
1,Switzerland,100.0,0.0,13118.00000
2,Norway,95.5,4.5,32551.00000
3,Sweden,91.4,8.6,26418.70000
4,Tajikistan,88.5,11.5,5296.40000
5,France,88.3,11.7,110615.92890
6,Brazil,84.7,15.2,147589.27133
7,New Zealand,82.6,17.4,6674.55000
8,Austria,81.6,18.4,11227.10000
9,Canada,77.8,22.2,143578.70000


In [21]:
query3 = """
SELECT fuel_category,
       ROUND(AVG(2024 - commissioning_year)::numeric, 1) AS avg_age_years,
       COUNT(*) AS plant_count
FROM power_plants
WHERE commissioning_year IS NOT NULL
GROUP BY fuel_category
ORDER BY avg_age_years DESC;
"""

pd.read_sql(query3, engine)

,fuel_category,avg_age_years,plant_count
0,Fossil,27.9,6291
1,Renewable,26.2,10374
2,Other,20.7,782


In [24]:
query4 = """
SELECT country_long,
       ROUND(AVG(2024 - commissioning_year)::numeric, 1) AS avg_age_years,
       COUNT(*) AS plant_count
FROM power_plants
WHERE commissioning_year IS NOT NULL
GROUP BY country_long
HAVING COUNT(*) > 20
ORDER BY avg_age_years DESC
LIMIT 10;
"""

pd.read_sql(query4, engine)

,country_long,avg_age_years,plant_count
0,Switzerland,73.7,150
1,France,68.4,66
2,Sweden,60.7,133
3,Russia,57.2,275
4,Austria,49.1,97
5,Czech Republic,45.9,29
6,Kazakhstan,45.4,23
7,Argentina,42.1,93
8,New Zealand,40.3,29
9,Germany,37.5,515
